In [1]:
import pandas as pd

onspd = pd.read_csv('../data/raw/london_postcodes-ons-postcodes-directory-feb22.csv',
                    nrows=5, low_memory=False)
print("Columns:", len(onspd.columns))
print(onspd.columns.tolist())

Columns: 50
['pcd', 'pcd2', 'pcds', 'dointr', 'doterm', 'oscty', 'ced', 'oslaua', 'osward', 'parish', 'usertype', 'oseast1m', 'osnrth1m', 'osgrdind', 'oshlthau', 'nhser', 'ctry', 'rgn', 'streg', 'pcon', 'eer', 'teclec', 'ttwa', 'pct', 'itl', 'statsward', 'oa01', 'casward', 'park', 'lsoa01', 'msoa01', 'ur01ind', 'oac01', 'oa11', 'lsoa11', 'msoa11', 'wz11', 'ccg', 'bua11', 'buasd11', 'ru11ind', 'oac11', 'lat', 'long', 'lep1', 'lep2', 'pfa', 'imd', 'calncv', 'stp']


## Spatial features from the ONS Postcode Directory

The prediction model requires continuous spatial features rather than 224
categorical district labels, so that it can generalise across sparsely
represented districts. This notebook derives a district centroid from the
easting and northing of every postcode in that district, computes distance
from central London, and records the Local Authority District needed for the
council tax join.

Source: ONS Postcode Directory for London, February 2022 release.
Output: `data/processed/district_spatial.csv`

In [2]:
# 50 columns in the file; load only the four required
cols = ['pcds', 'oseast1m', 'osnrth1m', 'oslaua']

onspd = pd.read_csv('../data/raw/london_postcodes-ons-postcodes-directory-feb22.csv',
                    usecols=cols, low_memory=False)

print(onspd.shape)
print(onspd.head())
print()
print("Missing coordinates:", onspd['oseast1m'].isna().sum())

(326214, 4)
      pcds     oslaua  oseast1m  osnrth1m
0  BR1 1AA  E09000006    540291    168873
1  BR1 1AB  E09000006    540262    169405
2  BR1 1AD  E09000006    540386    168710
3  BR1 1AE  E09000006    540197    169204
4  BR1 1AF  E09000006    540259    168855

Missing coordinates: 0


In [3]:
# The district is the part of the postcode before the space
onspd['district'] = onspd['pcds'].str.split().str[0]

# Mean coordinate of all postcodes in a district = its centroid
district = onspd.groupby('district').agg(
    easting=('oseast1m', 'mean'),
    northing=('osnrth1m', 'mean'),
    postcodes=('pcds', 'count'),
    lad_code=('oslaua', lambda s: s.mode().iloc[0]),
).reset_index()

print(district.shape)
print(district.head())

(329, 5)
  district        easting       northing  postcodes   lad_code
0      BR1  540543.442308  169901.966827       2080  E09000006
1      BR2  540757.997157  167642.069652       1407  E09000006
2      BR3  537000.145882  168984.669176       2125  E09000006
3      BR4  538607.253833  165934.688245        587  E09000006
4      BR5  546387.883295  167669.765828       1311  E09000006


### Distance from central London

Distance is measured from Charing Cross, the conventional reference point for
distances to London. Because easting and northing are metres on a flat
projection, straight-line distance is computed directly rather than requiring
spherical trigonometry.

In [4]:
import numpy as np

# Charing Cross is the conventional centre point for London distances
CENTRE_EASTING = 530034
CENTRE_NORTHING = 180381

district['dist_centre_km'] = np.sqrt(
    (district['easting'] - CENTRE_EASTING) ** 2 +
    (district['northing'] - CENTRE_NORTHING) ** 2
) / 1000

print(district[['district', 'dist_centre_km']]
      .sort_values('dist_centre_km')
      .head(10))
print()
print(district[['district', 'dist_centre_km']]
      .sort_values('dist_centre_km', ascending=False)
      .head(5))

    district  dist_centre_km
322     WC2N        0.315691
234     SW1Y        0.568167
320     WC2E        0.625813
321     WC2H        0.704967
227     SW1A        0.737344
324     WC99        0.763487
285      W1D        0.775318
298      W1V        0.791099
323     WC2R        0.840740
286      W1F        0.909737

    district  dist_centre_km
9       CM14       29.169386
8       CM13       28.851155
327      WD3       28.489650
169     RM14       27.118358
172      RM3       26.587260


In [5]:
out = district[['district', 'easting', 'northing', 'dist_centre_km',
                'lad_code', 'postcodes']]

out.to_csv('../data/processed/district_spatial.csv', index=False)
print("Saved:", out.shape)
print(out.head())

Saved: (329, 6)
  district        easting       northing  dist_centre_km   lad_code  postcodes
0      BR1  540543.442308  169901.966827       14.841109  E09000006       2080
1      BR2  540757.997157  167642.069652       16.651861  E09000006       1407
2      BR3  537000.145882  168984.669176       13.356779  E09000006       2125
3      BR4  538607.253833  165934.688245       16.798708  E09000006        587
4      BR5  546387.883295  167669.765828       20.712918  E09000006       1311


In [6]:
listings_districts = pd.read_csv('../data/processed/listings_clean.csv')['subdistrict_code'].unique() \
    if False else None

# Not saved yet, so check against the BRMA map instead
brma = pd.read_csv('../data/processed/district_brma.csv')
london_districts = brma[brma['brma'].str.contains('London', na=False)]['district']

missing = set(london_districts) - set(district['district'])
print("London BRMA districts with no ONSPD centroid:", len(missing))
print(sorted(missing)[:20] if missing else "none")

London BRMA districts with no ONSPD centroid: 1
['IG10']
